# Forschungsfrage 2 - Fehlende Werte: Moderner Non-LLM-Ansatz (Random-Forest-basierte Imputation nach dem MissForest-Prinzip)

## Methodische Vorbemerkung: von HyperImpute zu MissForest

Im ursprünglichen Untersuchungsplan war für diesen Abschnitt die AutoML-Bibliothek
**HyperImpute** (Jarrett et al., 2022) vorgesehen. Bei der praktischen Umsetzung
zeigte sich jedoch ein reproduzierbares Kompatibilitätsproblem mit den in der
Experimentierumgebung installierten aktuellen Versionen von scikit-learn/pydantic:
Der interne iterative Modellauswahl-Mechanismus von HyperImpute (`IterativeErrorCorrection`)
konvergierte nicht gegen ein trainiertes Vorhersagemodell, sondern gab für **jede**
fehlende Zeile ausschließlich den initialen Baseline-Wert (arithmetisches Mittel
bzw. häufigste Klasse) zurück - unabhängig vom gewählten Kandidatenmodell
(Random Forest, XGBoost, CatBoost, logistische Regression). Dies wurde durch
direkten Vergleich des zurückgegebenen Werts mit dem analytisch berechneten
Mittelwert der beobachteten Zielspalte verifiziert (exakte Übereinstimmung auf
10 Nachkommastellen) und äußerte sich in einer Testgenauigkeit von ca. 4 % bei
12 Klassen - deutlich unter dem Zufallsniveau von 8,3 %.

Als direkter Ersatz wird daher das **MissForest-Prinzip** (Stekhoven & Bühlmann,
2012) - eine der am häufigsten zitierten modernen, datengetriebenen
Non-LLM-Imputationsmethoden in der Literatur - über scikit-learns
`RandomForestRegressor`/`RandomForestClassifier` direkt implementiert. Ein Test mit
scikit-learns generischem `IterativeImputer`-Wrapper scheiterte ebenfalls, da dieser
konzeptionell **einen einzigen Schätzer-Typ für alle Spalten** der Eingabematrix
verwendet (auch für vollständig beobachtete Hilfsspalten) und dadurch bei gemischt
numerisch/kategorialen Datensätzen zwangsläufig scheitert (`ValueError: Unknown label
type: continuous` bei Verwendung eines Klassifikators auf eine numerische Spalte).

Es ist zu beachten, dass in dieser Untersuchung **keine vollständige iterative
MissForest-Implementierung** eingesetzt wird. Stattdessen wird das dem Verfahren
zugrunde liegende Random-Forest-Prinzip aufgabenspezifisch auf jeweils ein
Zielattribut angewendet: Für die numerische Variable `price` wird ein
`RandomForestRegressor`, für die kategoriale Variable `parent_category` ein
`RandomForestClassifier` trainiert. Die dabei verwendeten Hilfsmerkmale (u. a.
`parent_category`, `thread_category`, `source`, `saving`) weisen selbst teilweise
fehlende Werte auf - anders als im vollständigen MissForest-Algorithmus werden
diese vor dem Training **einmalig** gesondert behandelt (kategoriale
Platzhalterkategorie bzw. Median-Imputation auf Basis des Trainingssplits, siehe
Abschnitte 2 und 3) und nicht iterativ aus den jeweils aktuellen Modellschätzungen
der übrigen Spalten aktualisiert. Der Ansatz wird daher im Folgenden als
**Random-Forest-basierte Imputation nach dem MissForest-Prinzip** bezeichnet und
nicht als vollständige Implementierung des iterativen MissForest-Algorithmus
(Stekhoven & Bühlmann, 2012).

Bewertung ausschließlich auf dem `test`-Split - identisch zu den anderen beiden
TF2-Notebooks.


## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import json
import time
import os
import re

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, f1_score

SEED = 42
os.makedirs("results", exist_ok=True)


## 2. Numerisches Zielattribut: `price` (Random-Forest-Regression nach dem MissForest-Prinzip)

In [2]:
price_df = pd.read_csv("benchmark/tf2_missing_price.csv")
price_truth = pd.read_csv("benchmark/tf2_missing_price_groundtruth.csv").set_index("row_id")["price_true"]

cat_cols = ["parent_category", "thread_category", "source"]
num_cols = ["views", "votes", "replies"]

price_df[cat_cols] = price_df[cat_cols].fillna("__missing__")

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
train_mask = price_df["split"] == "train"
encoder.fit(price_df.loc[train_mask, cat_cols])

def build_X(df):
    cat_encoded = encoder.transform(df[cat_cols])
    return np.hstack([df[num_cols].values, cat_encoded])

X_train = build_X(price_df[train_mask])
y_train = price_df.loc[train_mask, "price_input"]

t0 = time.time()
rf_price = RandomForestRegressor(n_estimators=300, max_depth=None, random_state=SEED, n_jobs=-1)
rf_price.fit(X_train, y_train)
train_time_price = time.time() - t0

test_mask = price_df["split"] == "test"
X_test = build_X(price_df[test_mask])
test_ids = price_df.loc[test_mask, "row_id"].values
y_pred_price = rf_price.predict(X_test)
y_true_price = price_truth.loc[test_ids].values

mae = mean_absolute_error(y_true_price, y_pred_price)
rmse = np.sqrt(mean_squared_error(y_true_price, y_pred_price))
print(f"price - MAE: {mae:.3f}  RMSE: {rmse:.3f}  (n_test={len(test_ids)}, Trainingszeit {train_time_price:.2f}s)")

price_results = pd.DataFrame({"row_id": test_ids, "price_true": y_true_price, "price_pred_missforest": y_pred_price})
price_results.to_csv("results/tf2_missforest_price_predictions.csv", index=False)


price - MAE: 176.344  RMSE: 294.675  (n_test=164, Trainingszeit 1.53s)


## 3. Kategoriales Zielattribut: `parent_category` (Random-Forest-Klassifikation nach dem MissForest-Prinzip)

In [3]:
def rough_numeric(val):
    """Grobe numerische Näherung für price/saving-Rohstrings, nur als Feature-Signal
    (kein Ersatz für die eigentliche Formatbereinigung in Forschungsfrage 3)."""
    if pd.isna(val):
        return np.nan
    m = re.search(r"(\d+(?:\.\d+)?)", str(val))
    return float(m.group(1)) if m else np.nan

cat_df = pd.read_csv("benchmark/tf2_missing_parent_category.csv")
cat_truth = pd.read_csv("benchmark/tf2_missing_parent_category_groundtruth.csv").set_index("row_id")["parent_category_true"]

feature_cat_cols = ["thread_category", "source"]
feature_num_cols = ["views", "votes", "replies", "price", "saving"]

cat_df["price"] = cat_df["price"].apply(rough_numeric)
cat_df["saving"] = cat_df["saving"].apply(rough_numeric)

cat_df[feature_cat_cols] = cat_df[feature_cat_cols].fillna("__missing__")
train_mask2 = cat_df["split"] == "train"
_train_medians = cat_df.loc[train_mask2, feature_num_cols].median(numeric_only=True)
cat_df[feature_num_cols] = cat_df[feature_num_cols].fillna(_train_medians)

encoder2 = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder2.fit(cat_df.loc[train_mask2, feature_cat_cols])

def build_X2(df):
    cat_encoded = encoder2.transform(df[feature_cat_cols])
    return np.hstack([df[feature_num_cols].values, cat_encoded])

X_train2 = build_X2(cat_df[train_mask2])
y_train2 = cat_df.loc[train_mask2, "parent_category_input"]

t0 = time.time()
rf_cat = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1, class_weight="balanced")
rf_cat.fit(X_train2, y_train2)
train_time_cat = time.time() - t0

test_mask2 = cat_df["split"] == "test"
X_test2 = build_X2(cat_df[test_mask2])
test_ids2 = cat_df.loc[test_mask2, "row_id"].values
y_pred_cat = rf_cat.predict(X_test2)
y_true_cat = cat_truth.loc[test_ids2].values

acc = accuracy_score(y_true_cat, y_pred_cat)
macro_f1 = f1_score(y_true_cat, y_pred_cat, average="macro")
print(f"parent_category - Accuracy: {acc:.3f}  Macro-F1: {macro_f1:.3f}  (n_test={len(test_ids2)}, Trainingszeit {train_time_cat:.2f}s)")

cat_results = pd.DataFrame({"row_id": test_ids2, "parent_category_true": y_true_cat, "parent_category_pred_missforest": y_pred_cat})
cat_results.to_csv("results/tf2_missforest_parent_category_predictions.csv", index=False)


parent_category - Accuracy: 0.856  Macro-F1: 0.827  (n_test=125, Trainingszeit 0.50s)


## 4. Metriken und Laufzeit-Log speichern

In [4]:
metrics = {
    "experiment": "TF2_FehlendeWerte", "method": "RandomForest_nach_MissForest_Prinzip",
    "price_mae": mae, "price_rmse": rmse, "price_n_test": len(test_ids),
    "parent_category_accuracy": acc, "parent_category_macro_f1": macro_f1, "parent_category_n_test": len(test_ids2),
    "train_time_price_sec": train_time_price, "train_time_parent_category_sec": train_time_cat,
    "hinweis": "HyperImpute wegen Bibliotheks-Inkompatibilitaet (siehe Notebook-Einleitung) durch separate RandomForestRegressor-/RandomForestClassifier-Fits nach dem MissForest-Prinzip (Stekhoven & Buehlmann 2012) ersetzt - keine vollstaendige iterative MissForest-Implementierung.",
}
with open("results/tf2_missforest_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

log_rows = pd.DataFrame([
    {"experiment": "TF2_FehlendeWerte_price", "method": "RandomForest_nach_MissForest_Prinzip", "n_items": len(test_ids),
     "wall_time_sec": train_time_price, "input_tokens": 0, "output_tokens": 0,
     "estimated_cost_usd": 0.0, "model_name": "random_forest (MissForest-Prinzip, kein LLM/API)"},
    {"experiment": "TF2_FehlendeWerte_parent_category", "method": "RandomForest_nach_MissForest_Prinzip", "n_items": len(test_ids2),
     "wall_time_sec": train_time_cat, "input_tokens": 0, "output_tokens": 0,
     "estimated_cost_usd": 0.0, "model_name": "random_forest (MissForest-Prinzip, kein LLM/API)"},
])
log_path = "results/laufzeit_kosten_log.csv"
if os.path.exists(log_path):
    _old_log = pd.read_csv(log_path)
    _new_keys = set(zip(log_rows["experiment"], log_rows["method"]))
    _old_log = _old_log[~_old_log.apply(lambda r: (r["experiment"], r["method"]) in _new_keys, axis=1)]
    _combined_log = pd.concat([_old_log, log_rows], ignore_index=True)
else:
    _combined_log = log_rows
_combined_log.to_csv(log_path, index=False)

print("Gespeichert: results/tf2_missforest_price_predictions.csv, results/tf2_missforest_parent_category_predictions.csv, results/tf2_missforest_metrics.json")


Gespeichert: results/tf2_missforest_price_predictions.csv, results/tf2_missforest_parent_category_predictions.csv, results/tf2_missforest_metrics.json
